
# Data Dictionary



| Dataframe                             | Description                                                                                                                                |
| ------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------ |
| `combined_df_full_info`               | Full subject–phenotype dataset containing Manual and three LLM scores.                                                                     |
| `combined_df_full_info_adjudications` | Post-remapping nonunanimous subject–phenotype pairs with completed adjudication.                                                           |
| `gold_standard_full_info`             | Full dataset containing the constructed gold-standard score: unanimous score when all four methods agreed and adjudicated score otherwise. |





| Column                | Definition                                                                                                                |
| --------------------- | ------------------------------------------------------------------------------------------------------------------------- |
| `Subject ID`          | Unique identifier for each study subject.                                                                                 |
| `Phenotype`           | Phenotype evaluated for the subject.                                                                                      |
| `NOTE_TEXT`           | Full clinical-note text used to calculate note length.                                                                    |
| `NOTE_FILE_TIME_DTTM` | Date and time associated with the clinical note.                                                                          |
| `D.O.B.`              | Subject’s date of birth, used with note date to calculate age at note.                                                    |
| `Manual_Score`        | Manual phenotype classification: `1` = present and `0` = absent.                                                          |
| `GPT_Score`           | GPT 5.4 phenotype classification: `1` = present and `0` = absent.                                                         |
| `GPT_Mini_Score`      | GPT 5.4 Mini phenotype classification: `1` = present and `0` = absent.                                                    |
| `Claude_Sonnet_Score` | Claude Sonnet 4.6 phenotype classification: `1` = present and `0` = absent.                                               |
| `adjudicated_score`   | Adjudicator’s final phenotype classification: `1` = present and `0` = absent. Present only in the adjudication dataframe. |
| `gold_standard_score` | Constructed reference classification: `1` = present and `0` = absent. Present only in the gold-standard dataframe.        |


In [0]:
%pip install statsmodels
dbutils.library.restartPython()

In [0]:
import pandas as pd

# Define the base filepath
base_path = "/Workspace/Users/user/user/"

combined_df_full_info = pd.read_csv(f"{base_path}combined_df_full_info.csv")
display(combined_df_full_info)
print(len(combined_df_full_info))

combined_df_full_info_adjudications = pd.read_csv(f"{base_path}combined_df_full_info_adjudications.csv")
display(combined_df_full_info_adjudications)
print(len(combined_df_full_info_adjudications))

gold_standard_full_info = pd.read_csv(f"{base_path}gold_standard_full_info.csv")
display(gold_standard_full_info)
print(len(gold_standard_full_info))


In [0]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM


def prepare_predictors(dataframe):
    data = dataframe.copy()

    data["char_count"] = data["NOTE_TEXT"].astype("string").str.len()
    data["log_char_count"] = np.log1p(data["char_count"])

    data["age_at_note"] = (
        pd.to_datetime(data["NOTE_FILE_TIME_DTTM"]) - pd.to_datetime(data["D.O.B."])
    ).dt.days / 365.25

    data["note_time_numeric"] = (
        pd.to_datetime(data["NOTE_FILE_TIME_DTTM"])
        .dt.to_period("Q")
        .apply(lambda q: q.ordinal if pd.notna(q) else np.nan)
    )

    for column in ["log_char_count", "age_at_note", "note_time_numeric"]:
        data[f"z_{column}"] = (data[column] - data[column].mean()) / data[column].std()

    data["subject_id"] = data["Subject ID"].astype(str)
    data["phenotype_id"] = data["Phenotype"].astype(str)

    return data


def to_long_format(dataframe, method_columns, reference_column):
    pieces = []
    for display_name, column in method_columns.items():
        piece = dataframe.copy()
        piece["Method"] = display_name
        piece["disagreement"] = (piece[column] != piece[reference_column]).astype(int)
        pieces.append(piece)
    return pd.concat(pieces, ignore_index=True)


def fit_vb_with_retry(model, primary_maxiter=500, retry_maxiter=5000, retry_gtol=1e-4, rng=42):
    """
    Fits with standard settings first. If that doesn't converge, retries
    once, picking the retry based on why the first attempt stopped:
      - it hit the iteration cap while still genuinely moving (nit equals
        the cap): retry with a much higher cap, since it just ran out of
        budget.
      - it stopped early on its own, short of the cap (commonly a
        "precision loss" message): retry with a relaxed gradient
        tolerance instead, since more iterations alone won't change an
        already-settled answer.
    Either way, if the retry succeeds, its result replaces the original.
    """
    fit = model.fit_vb(minim_opts={"maxiter": primary_maxiter}, rng=rng)
    if fit.optim_retvals.success:
        return fit, "converged on the first attempt"

    if fit.optim_retvals.nit >= primary_maxiter:
        retry_fit = model.fit_vb(minim_opts={"maxiter": retry_maxiter}, rng=rng)
        retry_note = "ran out of iterations; retried with a higher cap"
    else:
        retry_fit = model.fit_vb(minim_opts={"maxiter": retry_maxiter, "gtol": retry_gtol}, rng=rng)
        retry_note = "stopped early; retried with a relaxed gradient tolerance"

    if retry_fit.optim_retvals.success:
        return retry_fit, f"{retry_note}, converged on retry"
    return retry_fit, f"{retry_note}, still did not converge"


def fit_pooled_stratum_model(long_dataframe, reference_column, reference_value, reference_method):
    required_columns = [
        "z_log_char_count", "z_age_at_note", "z_note_time_numeric",
        "Method", "disagreement", "subject_id", "phenotype_id",
    ]
    stratum = (
        long_dataframe.loc[long_dataframe[reference_column].eq(reference_value)]
        .dropna(subset=required_columns)
        .reset_index(drop=True)
    )

    formula = (
        f"disagreement ~ C(Method, Treatment(reference='{reference_method}')) "
        "+ z_log_char_count + z_age_at_note + z_note_time_numeric"
    )
    vc_formulas = {
        "Subject": "0 + C(subject_id)",
        "Phenotype": "0 + C(phenotype_id)",
    }

    model = BinomialBayesMixedGLM.from_formula(formula, vc_formulas, stratum, fe_p=2.0, vcp_p=0.5)
    fit, convergence_note = fit_vb_with_retry(model)

    z_score = fit.fe_mean / fit.fe_sd
    approx_p_value = 2 * norm.sf(np.abs(z_score))

    summary = pd.DataFrame({
        "Term": model.exog_names,
        "Log-odds": fit.fe_mean,
        "Odds ratio": np.exp(fit.fe_mean),
        "Posterior SD": fit.fe_sd,
        "Lower 95% CrI": np.exp(fit.fe_mean - 1.96 * fit.fe_sd),
        "Upper 95% CrI": np.exp(fit.fe_mean + 1.96 * fit.fe_sd),
        "Approx. p-value": approx_p_value,
    })
    summary["p-value (display)"] = summary["Approx. p-value"].apply(
        lambda p: "< 1e-300 (below floating-point precision)" if p == 0.0 else f"{p:.3g}"
    )

    return model, fit, stratum, summary, convergence_note


def check_convergence(fit, stratum, label):
    """
    Only relevant if the retry in fit_vb_with_retry still didn't converge.
    Checks for the most common structural cause: a phenotype or subject
    with 0% or 100% observed disagreement in this stratum, which is real,
    expected clinical signal, not a data error, but does make the
    optimization surface harder to fully settle on. Groups are split by
    row count, since a zero rate backed by many rows is trustworthy, while
    one backed by a handful of rows may just be too little data to tell
    zero from "rare but nonzero".
    """
    reports = {}
    for group_column in ["phenotype_id", "subject_id"]:
        group_rates = stratum.groupby(group_column)["disagreement"].agg(["mean", "count"])
        separated = group_rates[(group_rates["mean"] == 0) | (group_rates["mean"] == 1)]
        well_supported = separated[separated["count"] >= 10]
        thin = separated[separated["count"] < 10]
        print(
            f"  {group_column}: {len(separated)} of {len(group_rates)} groups are "
            f"all-agree or all-disagree ({len(well_supported)} backed by 10+ rows, "
            f"{len(thin)} backed by fewer)"
        )
        reports[group_column] = separated
    return reports


def verify_with_rescaling(model, fit, label, tolerance=0.10):
    """
    Last-resort check for a stratum that still hasn't converged after the
    retry: refit from a differently-conditioned starting point (scale_fe
    rescales the fixed-effect columns before optimizing). This only counts
    as confirmation when the comparison fit itself actually converges.
    """
    rescaled_fit = model.fit_vb(scale_fe=True, minim_opts={"maxiter": 5000}, rng=42)

    comparison = pd.DataFrame({
        "Term": model.exog_names,
        "Original odds ratio": np.exp(fit.fe_mean),
        "Rescaled odds ratio": np.exp(rescaled_fit.fe_mean),
    })
    comparison["Relative difference"] = (
        (comparison["Rescaled odds ratio"] - comparison["Original odds ratio"]).abs()
        / comparison["Original odds ratio"]
    )
    comparison["Stable"] = comparison["Relative difference"] < tolerance

    print(f"\n{label}: comparison against a rescaled refit")
    print(f"  Rescaled fit optimizer success = {rescaled_fit.optim_retvals.success}")

    if not rescaled_fit.optim_retvals.success:
        print("  The comparison fit also failed to converge, so agreement between the two doesn't confirm anything.")
        print("  Rely on cross-run coefficient stability instead (compare against earlier notebook runs of this stratum).")
    elif comparison["Stable"].all():
        print("  All fixed-effect terms agree within tolerance, and the comparison fit converged cleanly: treat the original fit as trustworthy.")
    else:
        unstable_terms = comparison.loc[~comparison["Stable"], "Term"].tolist()
        print(f"  These terms disagree beyond tolerance, treat with caution: {unstable_terms}")

    return comparison


def fit_and_report(long_dataframe, reference_column, reference_value, reference_method, label, results_dict):
    model, fit, stratum, summary, convergence_note = fit_pooled_stratum_model(
        long_dataframe, reference_column, reference_value, reference_method
    )
    print(f"{label}: {convergence_note} (iterations used = {fit.optim_retvals.nit})")

    separation_report = None
    rescale_comparison = None
    if not fit.optim_retvals.success:
        print(f"  Still unresolved after retry; checking for separated groups")
        separation_report = check_convergence(fit, stratum, label)
        rescale_comparison = verify_with_rescaling(model, fit, label)

    results_dict[reference_value] = {
        "model": model, "fit": fit, "stratum": stratum, "summary": summary,
        "convergence_note": convergence_note,
        "separation_report": separation_report, "rescale_comparison": rescale_comparison,
    }
    print(f"\n{label}")
    display(summary)


# Benchmark 1: three LLMs vs Manual score, full cohort

benchmark1_data = prepare_predictors(combined_df_full_info)
benchmark1_long = to_long_format(
    benchmark1_data,
    {"GPT 5.4": "GPT_Score", "GPT 5.4 Mini": "GPT_Mini_Score", "Claude Sonnet 4.6": "Claude_Sonnet_Score"},
    "Manual_Score",
)

benchmark1_results = {}
for manual_value in [1, 0]:
    fit_and_report(
        benchmark1_long, "Manual_Score", manual_value, "GPT 5.4",
        f"Benchmark 1, Manual score = {manual_value}", benchmark1_results,
    )


# Benchmark 2: Manual + three LLMs vs adjudicated score, restricted to the discordant subset

benchmark2_data = prepare_predictors(combined_df_full_info_adjudications)
benchmark2_long = to_long_format(
    benchmark2_data,
    {"Manual": "Manual_Score", "GPT 5.4": "GPT_Score", "GPT 5.4 Mini": "GPT_Mini_Score", "Claude Sonnet 4.6": "Claude_Sonnet_Score"},
    "adjudicated_score",
)

benchmark2_results = {}
for adjudicated_value in [1, 0]:
    fit_and_report(
        benchmark2_long, "adjudicated_score", adjudicated_value, "Manual",
        f"Benchmark 2, Adjudicated score = {adjudicated_value}", benchmark2_results,
    )

In [0]:
import re

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FixedFormatter, NullLocator
from scipy.stats import rankdata

# Run this after the model cell, so that benchmark1_results and
# benchmark2_results already exist.

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

covariate_labels = {
    "z_log_char_count": "Note length",
    "z_age_at_note": "Age at note",
    "z_note_time_numeric": "Note date"
}

# Order of the method rows, top to bottom. The reference method of a
# benchmark has no row of its own, so it is skipped automatically.
# Any method not listed here is placed after these, alphabetically.
method_order = [
    "Manual",
    "GPT 5.4",
    "GPT 5.4 Mini",
    "Claude Sonnet 4.6"
]

point_color = "#2F7BDC"
muted_color = "#666666"
grid_color = "#DDDDDD"

panel_letters = "ABCDEFGH"


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def compute_auc(result):
    """
    Area under the ROC curve for one fitted stratum, using the
    fixed-effects-only predicted probabilities (fit.predict() ignores
    the subject and phenotype random effects).
    """
    observed = np.asarray(result["model"].endog)
    predicted = np.asarray(result["fit"].predict())

    number_positive = (observed == 1).sum()
    number_negative = (observed == 0).sum()

    if number_positive == 0 or number_negative == 0:
        return np.nan

    # Rank-based (Mann-Whitney) AUC; ties receive average ranks
    ranks = rankdata(predicted)

    return (
        ranks[observed == 1].sum()
        - number_positive * (number_positive + 1) / 2
    ) / (number_positive * number_negative)


def build_forest_rows(summary, reference_method):
    """
    Turn a model summary table into the ordered rows of the plot:
    a method block, a blank line, then a covariate block.
    """
    method_rows = []
    covariate_rows = {}

    for _, term_row in summary.iterrows():

        term = term_row["Term"]

        estimate = {
            "odds_ratio": term_row["Odds ratio"],
            "lower": term_row["Lower 95% CrI"],
            "upper": term_row["Upper 95% CrI"]
        }

        method_match = re.search(r"\[T\.(.+)\]$", term)

        if method_match:
            method_rows.append(
                {"label": method_match.group(1), **estimate}
            )
        elif term in covariate_labels:
            covariate_rows[term] = {
                "label": covariate_labels[term], **estimate
            }

    method_rows = sorted(
        method_rows,
        key=lambda row: (
            method_order.index(row["label"])
            if row["label"] in method_order
            else len(method_order),
            row["label"]
        )
    )

    rows = [
        {"header": f"Method (vs {reference_method})"}
    ]
    rows += method_rows
    rows += [
        {"spacer": True},
        {"header": "Per 1 SD increase"}
    ]
    rows += [
        covariate_rows[term]
        for term in covariate_labels
        if term in covariate_rows
    ]

    return rows


def choose_axis_limits(rows):
    """Log-scale limits that cover every interval and the line at 1."""
    lowers = [row["lower"] for row in rows if "lower" in row]
    uppers = [row["upper"] for row in rows if "upper" in row]

    log_low = min(np.log(min(lowers)), 0)
    log_high = max(np.log(max(uppers)), 0)

    padding = 0.12 * (log_high - log_low)
    log_low -= padding
    log_high += padding

    # Keep some room on the shorter side of 1 so the reference line
    # never sits on the edge of the plot
    log_low = min(log_low, -0.45 * log_high)
    log_high = max(log_high, -0.45 * log_low)

    return np.exp(log_low), np.exp(log_high)


def choose_ticks(x_low, x_high):
    """Readable odds-ratio ticks, always including 1."""
    log_width = np.log(x_high / x_low)

    if x_high / x_low > 10:
        candidates = [
            0.001, 0.003, 0.01, 0.03, 0.1, 0.3,
            1, 3, 10, 30, 100, 300, 1000
        ]
    else:
        candidates = [
            0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.75,
            0.8, 0.9, 1, 1.1, 1.25, 1.5, 1.75, 2, 2.5, 3, 4, 5,
            6, 8, 10
        ]

    candidates = [
        value
        for value in candidates
        if x_low <= value <= x_high
    ]

    minimum_gap = 0.18 * log_width

    # Work outward from 1, skipping ticks that would crowd a neighbor
    ticks = [1]

    for value in sorted(
        [value for value in candidates if value < 1],
        reverse=True
    ):
        if np.log(ticks[0] / value) >= minimum_gap:
            ticks.insert(0, value)

    for value in [value for value in candidates if value > 1]:
        if np.log(value / ticks[-1]) >= minimum_gap:
            ticks.append(value)

    return ticks


def draw_forest_panel(ax, result, title, letter, reference_method):

    rows = build_forest_rows(
        result["summary"],
        reference_method
    )

    x_low, x_high = choose_axis_limits(rows)

    ax.set_xscale("log")
    ax.set_xlim(x_low, x_high)

    ticks = choose_ticks(x_low, x_high)

    ax.xaxis.set_major_locator(FixedLocator(ticks))
    ax.xaxis.set_major_formatter(
        FixedFormatter([f"{tick:g}" for tick in ticks])
    )
    ax.xaxis.set_minor_locator(NullLocator())

    # Light vertical guides at each tick, darker line at OR = 1
    for tick in ticks:
        if tick != 1:
            ax.axvline(
                tick,
                color=grid_color,
                linewidth=1,
                zorder=0
            )

    ax.axvline(
        1,
        color="#222222",
        linewidth=1.4,
        zorder=1
    )

    # y positions run top to bottom; x positions for the text columns
    # are in axes fractions so they stay put on a log axis
    text_transform = ax.get_yaxis_transform()

    for position, row in enumerate(rows):

        if "spacer" in row:
            continue

        if "header" in row:
            ax.text(
                -0.02,
                position,
                row["header"],
                transform=text_transform,
                ha="right",
                va="center",
                fontsize=14,
                fontweight="bold"
            )
            continue

        ax.plot(
            [row["lower"], row["upper"]],
            [position, position],
            color=point_color,
            linewidth=2.5,
            solid_capstyle="butt",
            zorder=2
        )

        ax.plot(
            row["odds_ratio"],
            position,
            "o",
            color=point_color,
            markersize=10,
            zorder=3
        )

        ax.text(
            -0.02,
            position,
            row["label"],
            transform=text_transform,
            ha="right",
            va="center",
            fontsize=15
        )

        ax.text(
            1.04,
            position,
            (
                f"{row['odds_ratio']:.3f} "
                f"({row['lower']:.3f} to {row['upper']:.3f})"
            ),
            transform=text_transform,
            ha="left",
            va="center",
            fontsize=14,
            fontfamily="monospace"
        )

    ax.set_ylim(len(rows) - 0.4, -0.6)
    ax.set_yticks([])

    for side in ["top", "left", "right"]:
        ax.spines[side].set_visible(False)

    ax.spines["bottom"].set_color("#888888")

    ax.tick_params(
        axis="x",
        labelsize=14,
        color="#888888"
    )

    ax.set_xlabel(
        "Odds ratio of disagreement (log scale)",
        fontsize=15,
        labelpad=8
    )

    # Direction hints and the column heading, just above the plot
    ax.text(
        0,
        1.03,
        "← fewer disagreements",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=12,
        color=muted_color
    )

    ax.text(
        1,
        1.03,
        "more disagreements →",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=12,
        color=muted_color
    )

    ax.text(
        1.03,
        1.03,
        "OR (95% CrI)",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=12,
        color=muted_color
    )

    # Panel letter, title, and AUC on one line above that
    ax.text(
        -0.52,
        1.16,
        letter,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=22,
        fontweight="bold"
    )

    ax.text(
        -0.45,
        1.16,
        title,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=17,
        fontweight="bold"
    )

    auc = compute_auc(result)

    ax.text(
        1.03,
        1.16,
        "AUC NA" if np.isnan(auc) else f"AUC {auc:.3f}",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=14,
        color=muted_color
    )

    return len(rows)


def plot_benchmark_forest(
    results_dict,
    title_template,
    reference_method,
    reference_values=(1, 0)
):
    """
    One stacked forest-plot panel per stratum.

    results_dict:     benchmark1_results or benchmark2_results
    title_template:   panel title with a {value} placeholder
    reference_method: the method the others are compared against
    """
    row_counts = [
        len(
            build_forest_rows(
                results_dict[value]["summary"],
                reference_method
            )
        )
        for value in reference_values
    ]

    # Layout in inches: plot height scales with the number of rows,
    # with fixed space above for the title lines and below for the
    # axis label
    axes_heights = [
        0.46 * count
        for count in row_counts
    ]

    top_space = 1.10
    bottom_space = 0.95
    gap_between_panels = 2.10

    figure_height = (
        sum(axes_heights)
        + top_space
        + bottom_space
        + gap_between_panels * (len(axes_heights) - 1)
    )

    fig, axes = plt.subplots(
        nrows=len(reference_values),
        ncols=1,
        figsize=(14, figure_height),
        gridspec_kw={
            "height_ratios": axes_heights,
            "left": 0.28,
            "right": 0.76,
            "top": 1 - top_space / figure_height,
            "bottom": bottom_space / figure_height,
            "hspace": gap_between_panels / np.mean(axes_heights)
        },
        squeeze=False
    )

    for index, (ax, value) in enumerate(
        zip(axes[:, 0], reference_values)
    ):
        draw_forest_panel(
            ax,
            results_dict[value],
            title_template.format(value=value),
            panel_letters[index],
            reference_method
        )

    return fig


# ------------------------------------------------------------
# Benchmark 1: three LLMs vs Manual score
# ------------------------------------------------------------

benchmark1_forest = plot_benchmark_forest(
    benchmark1_results,
    "Benchmark 1, Manual score = {value}",
    reference_method="GPT 5.4"
)

# benchmark1_forest.savefig("benchmark1_forest.png", dpi=300, bbox_inches="tight")

plt.show()

# ------------------------------------------------------------
# Benchmark 2: Manual + three LLMs vs adjudicated score
# ------------------------------------------------------------

benchmark2_forest = plot_benchmark_forest(
    benchmark2_results,
    "Benchmark 2, Adjudicated score = {value}",
    reference_method="Manual"
)

# benchmark2_forest.savefig("benchmark2_forest.png", dpi=300, bbox_inches="tight")

plt.show()

In [0]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM


# Settings for the gold standard benchmark. Adjust these if your column names differ.
GOLD_STANDARD_COLUMN = "gold_standard_score"
REFERENCE_METHOD = "Manual"
METHOD_COLUMNS = {
    "Manual": "Manual_Score",
    "GPT 5.4": "GPT_Score",
    "GPT 5.4 Mini": "GPT_Mini_Score",
    "Claude Sonnet 4.6": "Claude_Sonnet_Score",
}


def prepare_predictors(dataframe):
    data = dataframe.copy()

    data["char_count"] = data["NOTE_TEXT"].astype("string").str.len()
    data["log_char_count"] = np.log1p(data["char_count"])

    data["age_at_note"] = (
        pd.to_datetime(data["NOTE_FILE_TIME_DTTM"]) - pd.to_datetime(data["D.O.B."])
    ).dt.days / 365.25

    data["note_time_numeric"] = (
        pd.to_datetime(data["NOTE_FILE_TIME_DTTM"])
        .dt.to_period("Q")
        .apply(lambda q: q.ordinal if pd.notna(q) else np.nan)
    )

    for column in ["log_char_count", "age_at_note", "note_time_numeric"]:
        data[f"z_{column}"] = (data[column] - data[column].mean()) / data[column].std()

    data["subject_id"] = data["Subject ID"].astype(str)
    data["phenotype_id"] = data["Phenotype"].astype(str)

    return data


def to_long_format(dataframe, method_columns, reference_column):
    pieces = []
    for display_name, column in method_columns.items():
        piece = dataframe.copy()
        piece["Method"] = display_name
        piece["disagreement"] = (piece[column] != piece[reference_column]).astype(int)
        pieces.append(piece)
    return pd.concat(pieces, ignore_index=True)


def fit_vb_with_retry(model, primary_maxiter=500, retry_maxiter=5000, retry_gtol=1e-4, rng=42):
    """
    Fits with standard settings first. If that doesn't converge, retries
    once, picking the retry based on why the first attempt stopped:
      - it hit the iteration cap while still genuinely moving (nit equals
        the cap): retry with a much higher cap, since it just ran out of
        budget.
      - it stopped early on its own, short of the cap (commonly a
        "precision loss" message): retry with a relaxed gradient
        tolerance instead, since more iterations alone won't change an
        already-settled answer.
    Either way, if the retry succeeds, its result replaces the original.
    """
    fit = model.fit_vb(minim_opts={"maxiter": primary_maxiter}, rng=rng)
    if fit.optim_retvals.success:
        return fit, "converged on the first attempt"

    if fit.optim_retvals.nit >= primary_maxiter:
        retry_fit = model.fit_vb(minim_opts={"maxiter": retry_maxiter}, rng=rng)
        retry_note = "ran out of iterations; retried with a higher cap"
    else:
        retry_fit = model.fit_vb(minim_opts={"maxiter": retry_maxiter, "gtol": retry_gtol}, rng=rng)
        retry_note = "stopped early; retried with a relaxed gradient tolerance"

    if retry_fit.optim_retvals.success:
        return retry_fit, f"{retry_note}, converged on retry"
    return retry_fit, f"{retry_note}, still did not converge"


def fit_pooled_stratum_model(long_dataframe, reference_column, reference_value, reference_method):
    required_columns = [
        "z_log_char_count", "z_age_at_note", "z_note_time_numeric",
        "Method", "disagreement", "subject_id", "phenotype_id",
    ]
    stratum = (
        long_dataframe.loc[long_dataframe[reference_column].eq(reference_value)]
        .dropna(subset=required_columns)
        .reset_index(drop=True)
    )

    formula = (
        f"disagreement ~ C(Method, Treatment(reference='{reference_method}')) "
        "+ z_log_char_count + z_age_at_note + z_note_time_numeric"
    )
    vc_formulas = {
        "Subject": "0 + C(subject_id)",
        "Phenotype": "0 + C(phenotype_id)",
    }

    model = BinomialBayesMixedGLM.from_formula(formula, vc_formulas, stratum, fe_p=2.0, vcp_p=0.5)
    fit, convergence_note = fit_vb_with_retry(model)

    z_score = fit.fe_mean / fit.fe_sd
    approx_p_value = 2 * norm.sf(np.abs(z_score))

    summary = pd.DataFrame({
        "Term": model.exog_names,
        "Log-odds": fit.fe_mean,
        "Odds ratio": np.exp(fit.fe_mean),
        "Posterior SD": fit.fe_sd,
        "Lower 95% CrI": np.exp(fit.fe_mean - 1.96 * fit.fe_sd),
        "Upper 95% CrI": np.exp(fit.fe_mean + 1.96 * fit.fe_sd),
        "Approx. p-value": approx_p_value,
    })
    summary["p-value (display)"] = summary["Approx. p-value"].apply(
        lambda p: "< 1e-300 (below floating-point precision)" if p == 0.0 else f"{p:.3g}"
    )

    return model, fit, stratum, summary, convergence_note


def check_convergence(fit, stratum, label):
    """
    Only relevant if the retry in fit_vb_with_retry still didn't converge.
    Checks for the most common structural cause: a phenotype or subject
    with 0% or 100% observed disagreement in this stratum, which is real,
    expected clinical signal, not a data error, but does make the
    optimization surface harder to fully settle on. Groups are split by
    row count, since a zero rate backed by many rows is trustworthy, while
    one backed by a handful of rows may just be too little data to tell
    zero from "rare but nonzero".
    """
    reports = {}
    for group_column in ["phenotype_id", "subject_id"]:
        group_rates = stratum.groupby(group_column)["disagreement"].agg(["mean", "count"])
        separated = group_rates[(group_rates["mean"] == 0) | (group_rates["mean"] == 1)]
        well_supported = separated[separated["count"] >= 10]
        thin = separated[separated["count"] < 10]
        print(
            f"  {group_column}: {len(separated)} of {len(group_rates)} groups are "
            f"all-agree or all-disagree ({len(well_supported)} backed by 10+ rows, "
            f"{len(thin)} backed by fewer)"
        )
        reports[group_column] = separated
    return reports


def verify_with_rescaling(model, fit, label, tolerance=0.10):
    """
    Last-resort check for a stratum that still hasn't converged after the
    retry: refit from a differently-conditioned starting point (scale_fe
    rescales the fixed-effect columns before optimizing). This only counts
    as confirmation when the comparison fit itself actually converges.
    """
    rescaled_fit = model.fit_vb(scale_fe=True, minim_opts={"maxiter": 5000}, rng=42)

    comparison = pd.DataFrame({
        "Term": model.exog_names,
        "Original odds ratio": np.exp(fit.fe_mean),
        "Rescaled odds ratio": np.exp(rescaled_fit.fe_mean),
    })
    comparison["Relative difference"] = (
        (comparison["Rescaled odds ratio"] - comparison["Original odds ratio"]).abs()
        / comparison["Original odds ratio"]
    )
    comparison["Stable"] = comparison["Relative difference"] < tolerance

    print(f"\n{label}: comparison against a rescaled refit")
    print(f"  Rescaled fit optimizer success = {rescaled_fit.optim_retvals.success}")

    if not rescaled_fit.optim_retvals.success:
        print("  The comparison fit also failed to converge, so agreement between the two doesn't confirm anything.")
        print("  Rely on cross-run coefficient stability instead (compare against earlier notebook runs of this stratum).")
    elif comparison["Stable"].all():
        print("  All fixed-effect terms agree within tolerance, and the comparison fit converged cleanly: treat the original fit as trustworthy.")
    else:
        unstable_terms = comparison.loc[~comparison["Stable"], "Term"].tolist()
        print(f"  These terms disagree beyond tolerance, treat with caution: {unstable_terms}")

    return comparison


def fit_and_report(long_dataframe, reference_column, reference_value, reference_method, label, results_dict):
    model, fit, stratum, summary, convergence_note = fit_pooled_stratum_model(
        long_dataframe, reference_column, reference_value, reference_method
    )
    print(f"{label}: {convergence_note} (iterations used = {fit.optim_retvals.nit})")

    separation_report = None
    rescale_comparison = None
    if not fit.optim_retvals.success:
        print(f"  Still unresolved after retry; checking for separated groups")
        separation_report = check_convergence(fit, stratum, label)
        rescale_comparison = verify_with_rescaling(model, fit, label)

    results_dict[reference_value] = {
        "model": model, "fit": fit, "stratum": stratum, "summary": summary,
        "convergence_note": convergence_note,
        "separation_report": separation_report, "rescale_comparison": rescale_comparison,
    }
    print(f"\n{label}")
    display(summary)


# Gold standard benchmark: Manual + three LLMs vs gold standard score

gold_standard_data = prepare_predictors(gold_standard_full_info)
gold_standard_long = to_long_format(gold_standard_data, METHOD_COLUMNS, GOLD_STANDARD_COLUMN)

gold_standard_results = {}
for gold_value in [1, 0]:
    fit_and_report(
        gold_standard_long, GOLD_STANDARD_COLUMN, gold_value, REFERENCE_METHOD,
        f"Gold standard, gold standard score = {gold_value}", gold_standard_results,
    )

In [0]:
import re

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FixedFormatter, NullLocator
from scipy.stats import rankdata

# Run this after the gold standard model cell, so that
# gold_standard_results and REFERENCE_METHOD already exist.

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

covariate_labels = {
    "z_log_char_count": "Note length",
    "z_age_at_note": "Age at note",
    "z_note_time_numeric": "Note date"
}

# Order of the method rows, top to bottom. The reference method has
# no row of its own, so it is skipped automatically.
# Any method not listed here is placed after these, alphabetically.
method_order = [
    "Manual",
    "GPT 5.4",
    "GPT 5.4 Mini",
    "Claude Sonnet 4.6"
]

point_color = "#2F7BDC"
muted_color = "#666666"
grid_color = "#DDDDDD"

panel_letters = "ABCDEFGH"


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def compute_auc(result):
    """
    Area under the ROC curve for one fitted stratum, using the
    fixed-effects-only predicted probabilities (fit.predict() ignores
    the subject and phenotype random effects).
    """
    observed = np.asarray(result["model"].endog)
    predicted = np.asarray(result["fit"].predict())

    number_positive = (observed == 1).sum()
    number_negative = (observed == 0).sum()

    if number_positive == 0 or number_negative == 0:
        return np.nan

    # Rank-based (Mann-Whitney) AUC; ties receive average ranks
    ranks = rankdata(predicted)

    return (
        ranks[observed == 1].sum()
        - number_positive * (number_positive + 1) / 2
    ) / (number_positive * number_negative)


def build_forest_rows(summary, reference_method):
    """
    Turn a model summary table into the ordered rows of the plot:
    a method block, a blank line, then a covariate block.
    """
    method_rows = []
    covariate_rows = {}

    for _, term_row in summary.iterrows():

        term = term_row["Term"]

        estimate = {
            "odds_ratio": term_row["Odds ratio"],
            "lower": term_row["Lower 95% CrI"],
            "upper": term_row["Upper 95% CrI"]
        }

        method_match = re.search(r"\[T\.(.+)\]$", term)

        if method_match:
            method_rows.append(
                {"label": method_match.group(1), **estimate}
            )
        elif term in covariate_labels:
            covariate_rows[term] = {
                "label": covariate_labels[term], **estimate
            }

    method_rows = sorted(
        method_rows,
        key=lambda row: (
            method_order.index(row["label"])
            if row["label"] in method_order
            else len(method_order),
            row["label"]
        )
    )

    rows = [
        {"header": f"Method (vs {reference_method})"}
    ]
    rows += method_rows
    rows += [
        {"spacer": True},
        {"header": "Per 1 SD increase"}
    ]
    rows += [
        covariate_rows[term]
        for term in covariate_labels
        if term in covariate_rows
    ]

    return rows


def choose_axis_limits(rows):
    """Log-scale limits that cover every interval and the line at 1."""
    lowers = [row["lower"] for row in rows if "lower" in row]
    uppers = [row["upper"] for row in rows if "upper" in row]

    log_low = min(np.log(min(lowers)), 0)
    log_high = max(np.log(max(uppers)), 0)

    padding = 0.12 * (log_high - log_low)
    log_low -= padding
    log_high += padding

    # Keep some room on the shorter side of 1 so the reference line
    # never sits on the edge of the plot
    log_low = min(log_low, -0.45 * log_high)
    log_high = max(log_high, -0.45 * log_low)

    return np.exp(log_low), np.exp(log_high)


def choose_ticks(x_low, x_high):
    """Readable odds-ratio ticks, always including 1."""
    log_width = np.log(x_high / x_low)

    if x_high / x_low > 10:
        candidates = [
            0.001, 0.003, 0.01, 0.03, 0.1, 0.3,
            1, 3, 10, 30, 100, 300, 1000
        ]
    else:
        candidates = [
            0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.75,
            0.8, 0.9, 1, 1.1, 1.25, 1.5, 1.75, 2, 2.5, 3, 4, 5,
            6, 8, 10
        ]

    candidates = [
        value
        for value in candidates
        if x_low <= value <= x_high
    ]

    minimum_gap = 0.18 * log_width

    # Work outward from 1, skipping ticks that would crowd a neighbor
    ticks = [1]

    for value in sorted(
        [value for value in candidates if value < 1],
        reverse=True
    ):
        if np.log(ticks[0] / value) >= minimum_gap:
            ticks.insert(0, value)

    for value in [value for value in candidates if value > 1]:
        if np.log(value / ticks[-1]) >= minimum_gap:
            ticks.append(value)

    return ticks


def draw_forest_panel(ax, result, title, letter, reference_method):

    rows = build_forest_rows(
        result["summary"],
        reference_method
    )

    x_low, x_high = choose_axis_limits(rows)

    ax.set_xscale("log")
    ax.set_xlim(x_low, x_high)

    ticks = choose_ticks(x_low, x_high)

    ax.xaxis.set_major_locator(FixedLocator(ticks))
    ax.xaxis.set_major_formatter(
        FixedFormatter([f"{tick:g}" for tick in ticks])
    )
    ax.xaxis.set_minor_locator(NullLocator())

    # Light vertical guides at each tick, darker line at OR = 1
    for tick in ticks:
        if tick != 1:
            ax.axvline(
                tick,
                color=grid_color,
                linewidth=1,
                zorder=0
            )

    ax.axvline(
        1,
        color="#222222",
        linewidth=1.4,
        zorder=1
    )

    # y positions run top to bottom; x positions for the text columns
    # are in axes fractions so they stay put on a log axis
    text_transform = ax.get_yaxis_transform()

    for position, row in enumerate(rows):

        if "spacer" in row:
            continue

        if "header" in row:
            ax.text(
                -0.02,
                position,
                row["header"],
                transform=text_transform,
                ha="right",
                va="center",
                fontsize=14,
                fontweight="bold"
            )
            continue

        ax.plot(
            [row["lower"], row["upper"]],
            [position, position],
            color=point_color,
            linewidth=2.5,
            solid_capstyle="butt",
            zorder=2
        )

        ax.plot(
            row["odds_ratio"],
            position,
            "o",
            color=point_color,
            markersize=10,
            zorder=3
        )

        ax.text(
            -0.02,
            position,
            row["label"],
            transform=text_transform,
            ha="right",
            va="center",
            fontsize=15
        )

        ax.text(
            1.04,
            position,
            (
                f"{row['odds_ratio']:.3f} "
                f"({row['lower']:.3f} to {row['upper']:.3f})"
            ),
            transform=text_transform,
            ha="left",
            va="center",
            fontsize=14,
            fontfamily="monospace"
        )

    ax.set_ylim(len(rows) - 0.4, -0.6)
    ax.set_yticks([])

    for side in ["top", "left", "right"]:
        ax.spines[side].set_visible(False)

    ax.spines["bottom"].set_color("#888888")

    ax.tick_params(
        axis="x",
        labelsize=14,
        color="#888888"
    )

    ax.set_xlabel(
        "Odds ratio of disagreement (log scale)",
        fontsize=15,
        labelpad=8
    )

    # Direction hints and the column heading, just above the plot
    ax.text(
        0,
        1.03,
        "← fewer disagreements",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=12,
        color=muted_color
    )

    ax.text(
        1,
        1.03,
        "more disagreements →",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=12,
        color=muted_color
    )

    ax.text(
        1.03,
        1.03,
        "OR (95% CrI)",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=12,
        color=muted_color
    )

    # Panel letter, title, and AUC on one line above that
    ax.text(
        -0.52,
        1.16,
        letter,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=22,
        fontweight="bold"
    )

    ax.text(
        -0.45,
        1.16,
        title,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=17,
        fontweight="bold"
    )

    auc = compute_auc(result)

    ax.text(
        1.03,
        1.16,
        "AUC NA" if np.isnan(auc) else f"AUC {auc:.3f}",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=14,
        color=muted_color
    )

    return len(rows)


def plot_benchmark_forest(
    results_dict,
    title_template,
    reference_method,
    reference_values=(1, 0)
):
    """
    One stacked forest-plot panel per stratum.

    results_dict:     gold_standard_results
    title_template:   panel title with a {value} placeholder
    reference_method: the method the others are compared against
    """
    row_counts = [
        len(
            build_forest_rows(
                results_dict[value]["summary"],
                reference_method
            )
        )
        for value in reference_values
    ]

    # Layout in inches: plot height scales with the number of rows,
    # with fixed space above for the title lines and below for the
    # axis label
    axes_heights = [
        0.46 * count
        for count in row_counts
    ]

    top_space = 1.10
    bottom_space = 0.95
    gap_between_panels = 2.10

    figure_height = (
        sum(axes_heights)
        + top_space
        + bottom_space
        + gap_between_panels * (len(axes_heights) - 1)
    )

    fig, axes = plt.subplots(
        nrows=len(reference_values),
        ncols=1,
        figsize=(14, figure_height),
        gridspec_kw={
            "height_ratios": axes_heights,
            "left": 0.28,
            "right": 0.76,
            "top": 1 - top_space / figure_height,
            "bottom": bottom_space / figure_height,
            "hspace": gap_between_panels / np.mean(axes_heights)
        },
        squeeze=False
    )

    for index, (ax, value) in enumerate(
        zip(axes[:, 0], reference_values)
    ):
        draw_forest_panel(
            ax,
            results_dict[value],
            title_template.format(value=value),
            panel_letters[index],
            reference_method
        )

    return fig


# ------------------------------------------------------------
# Gold standard: Manual + three LLMs vs gold standard score
# ------------------------------------------------------------

gold_standard_forest = plot_benchmark_forest(
    gold_standard_results,
    "Gold standard score = {value}",
    reference_method=REFERENCE_METHOD
)

# gold_standard_forest.savefig("gold_standard_forest.png", dpi=300, bbox_inches="tight")

plt.show()